# 25 — Serving Models: vLLM, Batching, KV Cache, Latency, and Cost

**Network LLM Engineering — Part VI — Production Network AI**

### Learning goals
- Understand inference-serving metrics
- Run an OpenAI-compatible local server pattern
- Estimate memory/capacity tradeoffs

## Serving terminology

- **TTFT:** time to first token.
- **token throughput:** tokens/sec.
- **KV cache:** cached attention keys/values for previous tokens.
- **batching:** process requests together.
- **continuous batching:** dynamically mix active requests.
- **prefix caching:** reuse shared prompt-prefix computation.
- **tensor parallelism:** split model computation across GPUs.

In [ ]:
# Simple capacity arithmetic.
gpu_mem_gb = 80
model_weights_gb = 16
runtime_overhead_gb = 8
kv_budget = gpu_mem_gb - model_weights_gb - runtime_overhead_gb
print("Approx KV/activation/cache budget:", kv_budget, "GB")

## vLLM deployment pattern

A serving engine can expose an OpenAI-compatible API so your application code is decoupled from model-loading details.

Example shell command (adapt model and flags to your hardware):

```bash
vllm serve Qwen/Qwen3-8B   --host 0.0.0.0   --port 8000
```

Then an application can send chat-completion requests to the local endpoint.

In [ ]:
# Example client code; requires a running local compatible server.
example = r'''
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
resp = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[{"role":"user","content":"Explain EVPN Type 2 in two bullets."}],
    temperature=0,
)
print(resp.choices[0].message.content)
'''
print(example)

## Capacity is workload-specific

The same model can have very different throughput depending on:
- prompt length,
- output length,
- concurrent users,
- quantization,
- GPU type,
- context window,
- batching,
- reasoning-token usage.

Benchmark your real NOC prompt distribution, not a generic tokens/sec headline.

### Exercise

Define three SLOs:
- interactive engineer assistant,
- background ticket summarizer,
- event-driven triage.

Give each a TTFT/latency and throughput target.